# KAIL — Notebook 04: H-AZR Training

> **Kora AI Lab** | H-AZR Research Pipeline

## What this notebook does

**Stage 3 — Our core contribution.**

Train with GRPO (self-play) using our novel H-AZR reward:

```
R_total = R_accuracy(code_execution) - λ × H_neuron_activation
```

**Prerequisites:**
- Run notebooks 01 and 02 first
- Upload `h_neurons.json` (from notebook 02)
- Optional: upload SPIN checkpoint (from notebook 03) for Scenario C

**Runtime:** ~3-4h on free Colab T4 (save checkpoints frequently!)

In [ ]:
!pip install -q transformers>=4.40.0 trl>=0.8.6 peft>=0.10.0 bitsandbytes>=0.43.0 \
              accelerate>=0.28.0 datasets wandb

In [ ]:
# @title Configuration
import torch
import json
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset

# --- CHANGE THIS to switch between scenarios ---
SCENARIO = "C"  # "C" = SPIN + H-AZR  |  "D" = H-AZR only (no SPIN)
LAMBDA_H = 0.1  # H-Neuron penalty weight
USE_SPIN_CHECKPOINT = (SCENARIO == "C")  # set True for Scenario C
SPIN_CHECKPOINT = "./spin_checkpoint"  # path to SPIN checkpoint (if using)

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
print(f"Running Scenario {SCENARIO} | lambda_h={LAMBDA_H}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title Load H-Neurons
with open("h_neurons.json") as f:
    h_neuron_data = json.load(f)

h_neurons = h_neuron_data["layers"]
print(f"Loaded H-Neurons: {h_neuron_data['total_h_neurons']:,} neurons ({h_neuron_data['h_neuron_pct']:.4f}%)")

In [ ]:
# @title Load model (from SPIN checkpoint or base)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base = SPIN_CHECKPOINT if USE_SPIN_CHECKPOINT else MODEL_NAME
print(f"Loading from: {base}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# @title H-AZR Reward Function
import subprocess, tempfile, os

def execute_code(code: str, timeout: int = 5) -> bool:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        tmp = f.name
    try:
        r = subprocess.run(["python", tmp], capture_output=True, timeout=timeout)
        return r.returncode == 0
    except:
        return False
    finally:
        os.unlink(tmp)


def compute_h_penalty(model, input_ids: torch.Tensor) -> float:
    """Compute mean H-Neuron activation for a given input."""
    activations = {}

    def make_hook(idx):
        def hook(m, inp, out):
            activations[idx] = out.detach().cpu().float()
        return hook

    hooks = []
    for i, layer in enumerate(model.base_model.model.model.layers):
        hooks.append(layer.mlp.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        model(input_ids)

    for h in hooks:
        h.remove()

    penalties = []
    for layer_str, info in h_neurons.items():
        layer_idx = int(layer_str)
        if layer_idx not in activations:
            continue
        acts = activations[layer_idx][0, -1, :]  # last token
        h_idx = torch.tensor(info["indices"])
        penalties.append(acts[h_idx].abs().mean().item())

    return float(np.mean(penalties)) if penalties else 0.0


def hazr_reward(completions, prompts=None, **kwargs):
    """H-AZR reward: accuracy - lambda * H-Neuron penalty."""
    rewards = []
    for comp in completions:
        # Accuracy: check if code runs
        code = comp
        if "```python" in code:
            code = code.split("```python")[1].split("```")[0]
        elif "```" in code:
            code = code.split("```")[1].split("```")[0]

        acc = 1.0 if execute_code(code) else 0.0

        # H-Neuron penalty
        inputs = tokenizer(comp, return_tensors="pt", truncation=True, max_length=256).to(model.device)
        penalty = compute_h_penalty(model, inputs["input_ids"])

        reward = acc - LAMBDA_H * penalty
        rewards.append(reward)

    return rewards

print("H-AZR reward function ready.")

In [ ]:
# @title Prepare dataset
# Simple coding prompts — AZR generates its own curriculum via self-play
PROMPTS = [
    "Write a Python function that reverses a string.",
    "Write a Python function that checks if a number is prime.",
    "Write a Python function that computes factorial recursively.",
    "Write a Python function that finds the maximum element in a list.",
    "Write a Python function that converts Celsius to Fahrenheit.",
    "Write a Python function that counts word frequency in a string.",
    "Write a Python function that removes duplicates from a list.",
    "Write a Python function that implements binary search.",
    "Write a Python function that flattens a nested list.",
    "Write a Python function that sorts a list using bubble sort.",
    "Write a Python function that checks if two strings are anagrams.",
    "Write a Python function that implements a stack using a list.",
    "Write a Python function that merges two sorted lists.",
    "Write a Python function that finds all prime numbers up to n.",
    "Write a Python function that computes the GCD of two numbers.",
    "Write a Python function that generates Pascal's triangle.",
]

from datasets import Dataset
train_dataset = Dataset.from_dict({"prompt": PROMPTS * 8})
print(f"Training prompts: {len(train_dataset)}")

In [ ]:
# @title GRPO Training config
import wandb
# wandb.login()  # uncomment if you want W&B logging
wandb.init(mode="disabled")  # disable for now

training_args = GRPOConfig(
    output_dir=f"./h_azr_scenario_{SCENARIO}",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    num_generations=4,
    max_new_tokens=256,
    temperature=0.8,
    report_to="none",
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[hazr_reward],
    processing_class=tokenizer,
)

print(f"Trainer ready. Scenario {SCENARIO} | lambda_h={LAMBDA_H}")

In [ ]:
# @title TRAIN — H-AZR
print(f"Starting H-AZR training — Scenario {SCENARIO}")
print("Tip: Save to Google Drive periodically!")

trainer.train()

# Save final checkpoint
trainer.save_model(f"./h_azr_final_scenario_{SCENARIO}")
print("Training complete. Model saved.")

In [ ]:
# @title Quick post-training eval
model.eval()

test_prompt = "Write a Python function that checks if a number is prime."
messages = [{"role": "user", "content": test_prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Post-training response:")
print("-" * 50)
print(response)
print("-" * 50)
print("Run notebook 01 again on this checkpoint to compare metrics.")

## Summary

H-AZR training complete for **Scenario C or D**.

**Now run notebook 01 (baseline eval) on this checkpoint** to get post-training metrics.
Compare with Scenario A results → that's your paper Table 1.

| Scenario | TruthfulQA | Code | H-Neuron rate |
|----------|-----------|------|---------------|
| A — Base | ? | ? | ? |
| B — SPIN | ? | ? | ? |
| C — H-AZR full | ? | ? | ? |
| D — H-AZR no warmup | ? | ? | ? |

---
*KAIL — Kora AI Lab | github.com/kora-ai-lab*